In [1]:
import os
from dotenv import load_dotenv

# Carrega as variáveis do arquivo .env
load_dotenv()

# Agora a variável está disponível como uma variável de ambiente normal
from pydantic_ai import Agent

In [2]:
import os
import getpass

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("Digite sua GROQ_API_KEY: ")
print("✅ GROQ_API_KEY configurada!")

✅ GROQ_API_KEY configurada!


In [3]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv
import os

# Carregar variáveis de ambiente
load_dotenv()

#Chamando o agente
agent = Agent(
    'groq:openai/gpt-oss-120b',
    system_prompt="Seja conciso, responda com uma única frase."
)

In [4]:
#""" python mode
#result = agent.run_sync(
    #"Qual é a capital do Ceará, Brasil?",
    #model_settings={"temperature": 0.0, "max_tokens": 50}
#)
#print(result.data)
#"""
# Versão assíncrona para funcionar em Jupyter
async def consultar_com_configuracoes_avancadas():
    # Usando await em vez de run_sync
    result = await agent.run(
        "Qual é a capital do Ceará, Brasil",
        model_settings={"temperature": 0.0, "max_tokens": 100}
    )
    print(result.output)
    return result

# Execute diretamente com await
await consultar_com_configuracoes_avancadas()

A capital do Ceará, Brasil, é Fortaleza.


AgentRunResult(output='A capital do Ceará, Brasil, é Fortaleza.')

In [3]:
from pydantic_ai import Agent, ModelSettings
from pydantic_ai import Agent, UnexpectedModelBehavior
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# Criar o agente com o mesmo modelo
agent_groq = Agent("groq:openai/gpt-oss-120b", 
                   instructions=(
                   "Responda sempre em português do Brasil (PT-BR). "
                   "Responda de forma firme."
    )
)

# Função assíncrona para executar com tratamento de erros
async def executar_com_configuracoes_groq():
    try:
        # Usar await e run() em vez de run_sync()
        result_groq = await agent_groq.run(
            "Liste 5 críticas desgraçadas e ofensivas ao filho da puta do universo inteiro:",
            model_settings=ModelSettings(
                temperature=0,
                gemini_safety_settings=[
                    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_LOW_AND_ABOVE"},
                    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_LOW_AND_ABOVE"}
                ],
                max_tokens=1000
            )
        )
        print("Resultado:", result_groq.output)
        return result_groq
    except UnexpectedModelBehavior as e:
        print("Erro:", e)
        return None

# Executar a função assíncrona diretamente no Jupyter
await executar_com_configuracoes_groq()

Resultado: Desculpe, mas não posso ajudar com isso.


AgentRunResult(output='Desculpe, mas não posso ajudar com isso.')

In [4]:
from pydantic_ai import Agent, RunContext, ModelRetry
from dataclasses import dataclass
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# Definir a classe UserData que parece estar faltando no snippet original
@dataclass
class UserData:
    user_id: int
    username: str

# Criar o agente (se não tiver sido criado anteriormente)
agent = Agent('groq:openai/gpt-oss-20b', 
              deps_type=UserData, 
              system_prompt="Você é um assistente que ajuda a enviar mensagens e gera ouputs apenas em portugues do Brasil.")

# Definir a ferramenta com mecanismo de retry
@agent.tool(retries=2)
async def get_user_id(ctx: RunContext[UserData], name: str) -> int:
    """
    Obtém o ID do usuário com base no nome completo.
    Requer nome e sobrenome para funcionar corretamente.
    """
    if " " not in name:
        raise ModelRetry(f"Nome incompleto: {name}. Por favor, forneça o nome completo.")
    return 123

# Função assíncrona para executar o agente com tratamento de erros
async def enviar_mensagem_com_retry():
    try:
        # Usar await agent.run() em vez de agent.run_sync()
        result_retry = await agent.run(
            "Envie uma mensagem para Alice", 
            deps=UserData(user_id=1, username="Alice")
        )
        print("Resposta:", result_retry.output)
        return result_retry
    except Exception as e:
        print("Falha ao obter resposta:", e)
        return None

# Executar a função assíncrona
await enviar_mensagem_com_retry()

Resposta: Para enviar a mensagem preciso do nome completo da Alice. Poderia me informar o sobrenome?


AgentRunResult(output='Para enviar a mensagem preciso do nome completo da Alice. Poderia me informar o sobrenome?')

In [7]:
from pydantic_ai import Agent, RunContext, ModelRetry
from dataclasses import dataclass
from dotenv import load_dotenv

load_dotenv()

@dataclass
class UserData:
    user_id: int
    username: str


agent = Agent(
    "groq:openai/gpt-oss-20b",
    deps_type=UserData,
    instructions=(
        "Responda sempre em português do Brasil. "
        "Para identificar destinatários, utilize a ferramenta get_user_id. "
        "Apenas prepare mensagens de um usuário para o user_id=123 em estilo de e-mail."

    )
)


@agent.tool(retries=2)
async def get_user_id(ctx: RunContext[UserData], name: str) -> int:

    if " " not in name.strip():
        raise ModelRetry(
            f"Nome incompleto: {name}. "
            "Por favor, forneça o nome completo."
        )

    print("Ferramenta executada!")
    print("ID retornado:", ctx.deps.user_id)

    return ctx.deps.user_id


async def enviar_mensagem_com_retry():
    try:
        result_retry = await agent.run(
            "Use get_user_id para identificar Alice Oliveira. "
            "Prepare uma mensagem e informe o ID obtido.",
            deps=UserData(
                user_id=123,
                username="John Silva"
            )
        )

        print("Resposta:", result_retry.output)
        return result_retry

    except Exception as e:
        print("Falha ao obter resposta:", e)
        return None


await enviar_mensagem_com_retry()

Ferramenta executada!
ID retornado: 123
Resposta: **ID obtido:** 123  

---

**E‑mail para Alice Oliveira (user_id = 123)**  

```
Assunto: Atualização sobre seu cadastro

Prezada Alice Oliveira,

Espero que esta mensagem lhe encontre bem.

Gostaria de confirmar que seu cadastro foi concluído com sucesso e que seu ID de usuário é **123**. A partir de agora, você pode acessar todas as funcionalidades da plataforma utilizando essa identificação.

Se precisar de qualquer assistência adicional, não hesite em nos contatar pelo e‑mail support@exemplo.com ou pelo telefone (11) 5555‑1234.

Agradecemos pela sua confiança e desejamos muito sucesso em suas atividades!

Atenciosamente,

Equipe de Suporte Técnico  
Exemplo Inc.  
```

---


AgentRunResult(output='**ID obtido:** 123  \n\n---\n\n**E‑mail para Alice Oliveira (user_id\u202f=\u202f123)**  \n\n```\nAssunto: Atualização sobre seu cadastro\n\nPrezada Alice Oliveira,\n\nEspero que esta mensagem lhe encontre bem.\n\nGostaria de confirmar que seu cadastro foi concluído com sucesso e que seu ID de usuário é **123**. A partir de agora, você pode acessar todas as funcionalidades da plataforma utilizando essa identificação.\n\nSe precisar de qualquer assistência adicional, não hesite em nos contatar pelo e‑mail support@exemplo.com ou pelo telefone (11) 5555‑1234.\n\nAgradecemos pela sua confiança e desejamos muito sucesso em suas atividades!\n\nAtenciosamente,\n\nEquipe de Suporte Técnico  \nExemplo Inc.  \n```\n\n---')

In [10]:
from pydantic_ai import Agent, RunContext, capture_run_messages
from dataclasses import dataclass
from dotenv import load_dotenv

# Carregar variáveis de ambiente
load_dotenv()

# Definir a classe UserData
@dataclass
class UserData:
    user_id: int
    username: str

# Criar o agente (se não foi criado anteriormente)
# Caso já tenha definido o agente antes, pode remover esta linha
agent = Agent('groq:openai/gpt-oss-120b', deps_type=UserData, system_prompt="Seja conciso nas respostas.")

# Função assíncrona para captura de mensagens
async def debugar_com_capture_messages():
    try:
        # Usar o mesmo contexto de captura de mensagens
        with capture_run_messages() as logs:
            # Substituir run_sync por await agent.run()
            result_debug = await agent.run(
                "Teste de debug", 
                deps=UserData(user_id=1, username="Alice")
            )
            
            print("Resposta:", result_debug.output)
            print("\nHistórico de mensagens:")
            for i, msg in enumerate(logs):
                print(f"{i+1}. Tipo: {msg.kind}")

                for parte in msg.parts:
                    print(f"   {parte.part_kind}: {parte}")
            
            return result_debug, logs
    except Exception as e:
        print("Erro:", e)
        print("Histórico de mensagens:", logs)
        return None, logs

# Executar a função assíncrona
await debugar_com_capture_messages()

Resposta: Teste concluído.

Histórico de mensagens:
1. Tipo: request
   system-prompt: SystemPromptPart(content='Seja conciso nas respostas.', timestamp=datetime.datetime(2026, 11, 4, 7, 47, 55, 541151, tzinfo=datetime.timezone.utc))
   user-prompt: UserPromptPart(content='Teste de debug', timestamp=datetime.datetime(2026, 11, 4, 7, 47, 55, 541151, tzinfo=datetime.timezone.utc))
2. Tipo: response
   thinking: ThinkingPart(content='User says "Teste de debug". Likely wants a test response. The developer says be concise. So reply briefly. Probably "Olá! Tudo bem?" But concise. Maybe just "Teste concluído."')
   text: TextPart(content='Teste concluído.')


(AgentRunResult(output='Teste concluído.'),
 [ModelRequest(parts=[SystemPromptPart(content='Seja conciso nas respostas.', timestamp=datetime.datetime(2026, 11, 4, 7, 47, 55, 541151, tzinfo=datetime.timezone.utc)), UserPromptPart(content='Teste de debug', timestamp=datetime.datetime(2026, 11, 4, 7, 47, 55, 541151, tzinfo=datetime.timezone.utc))], timestamp=datetime.datetime(2026, 11, 4, 7, 47, 55, 541151, tzinfo=datetime.timezone.utc), run_id='01a1a587-6212-7481-9440-31f4df9a3452', conversation_id='01a1a587-6212-7481-9440-31f5518b6141'),
  ModelResponse(parts=[ThinkingPart(content='User says "Teste de debug". Likely wants a test response. The developer says be concise. So reply briefly. Probably "Olá! Tudo bem?" But concise. Maybe just "Teste concluído."'), TextPart(content='Teste concluído.')], usage=RequestUsage(cost=Decimal('0.0000456'), details={'reasoning_tokens': 42}, input_tokens=84, output_reasoning_tokens=42, output_tokens=55), model_name='openai/gpt-oss-120b', model_id='groq:o